In [4]:
import cmocean
import numpy as np 
import xarray as xr
import pandas as pd 
import seaborn as sns
import cartopy.crs as ccrs
import statsmodels.api as sm
import matplotlib.pyplot as plt
from IPython.display import HTML
from scipy.stats import linregress 
from nemo_cookbook import NEMODataTree
import cartopy.feature as cfeature
from matplotlib.patches import Rectangle
from OceanDataStore import OceanDataCatalog 
from matplotlib.animation import FuncAnimation


In [5]:
# -- Initialise Dask Local Cluster -- #
import os
import dask
from dask.distributed import Client, LocalCluster

# Update temporary directory for Dask workers:
dask.config.set({'temporary_directory': f"{os.getcwd()}/dask_tmp",
                 'local_directory': f"{os.getcwd()}/dask_tmp"
                 })

# Create Local Cluster:
cluster = LocalCluster(n_workers=4, threads_per_worker=3, memory_limit='5GB')
client = Client(cluster)
client


C:\Users\TomAH\anaconda3\envs\env_ods\Lib\site-packages\distributed\node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 49675 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:49675/status,
Dashboard: http://127.0.0.1:49675/status,Workers: 4
Total threads: 12,Total memory: 18.63 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:49676,Workers: 0
Dashboard: http://127.0.0.1:49675/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:57191,Total threads: 3
Dashboard: http://127.0.0.1:49793/status,Memory: 4.66 GiB
Nanny: tcp://127.0.0.1:49679,


In [6]:
# Define directory path to ancillary files:
domain_filepath = "https://noc-msm-o.s3-ext.jc.rl.ac.uk/npd-eorca1-jra55v1/domain"

# Open eORCA1 NEMO model domain_cfg:
ds_domain = (xr.open_zarr(f"{domain_filepath}/domain_cfg", consolidated=True, chunks={})
             .squeeze()
             .rename({'z': 'nav_lev'})
             )
# Open eORCA1 ocean basin masks:
ds_subbasins = xr.open_zarr(f"{domain_filepath}/subbasins", consolidated=True, chunks={})

ds_domain

GroupNotFoundError: No group found in store 'https://noc-msm-o.s3-ext.jc.rl.ac.uk/npd-eorca1-jra55v1/domain/domain_cfg' at path ''

In [4]:
# Define directory path to model output files:
output_dir = "https://noc-msm-o.s3-ext.jc.rl.ac.uk/npd-eorca1-jra55v1/V1y"

# Construct NEMO model grid dataset, including vertical grid cell thicknesses (m) and meridional velocities (m/s):
ds_gridV = xr.merge([xr.open_zarr(f"{output_dir}/{var}", consolidated=True, chunks={})[var] for var in ['e3v', 'vo']], compat="override")

ds_gridV

ValueError: Consolidated metadata requested with 'use_consolidated=True' but not found in ''.

In [ ]:
# Define dictionary of grid datasets defining eORCA1 parent model domain with no child/grand-child nests:
# Note: domain_cfg z-dimension is expected to be named 'nav_lev'.
datasets = {"parent": {"domain": ds_domain, "gridV": ds_gridV}}

# Initialise a new NEMODataTree whose parent domain is zonally periodic & north-folding on F-points:
nemo = NEMODataTree.from_datasets(datasets=datasets, iperio=True, nftype="F")

nemo

In [ ]:
# Define Atlantic Ocean basin mask:
atlmask = ds_subbasins['atlmsk'].rename({"x":"i", "y":"j"}).astype(bool)
# Assign (i,j) coordinates of V-points:
atlmask['i'] = atlmask['i'] + 1
atlmask['j'] = atlmask['j'] + 1.5

# Compute barotropic stream function:
bsf_atl = nemo.integral(grid="gridV",
                        var="vo",
                        dims=["k", "i"], 
                        cum_dims=["i"],
                        dir="+1",
                        mask=atlmask
                        )

bsf_atl

In [ ]:
# Compute barotropic stream function in Sverdrups [1 Sv = 1E6 m3/s]:
bsf_atl = 1E-6 * bsf_atl.compute()

In [ ]:
bsf_atl = bsf_atl.sel(time_counter = slice(np.datetime64('1990-01-01'), np.datetime64('2024-12-31')))

In [ ]:
## Plotting Animation of annual BSF

plt.close('all')
fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(10, 6))

im = ax.pcolormesh(bsf_atl['glamv'], bsf_atl['gphiv'], bsf_atl.isel(time_counter = 0), transform=ccrs.PlateCarree(),  cmap="RdBu_r", shading="auto", vmin= -30, vmax=30, zorder = 1)  
cont = ax.contour(bsf_atl['glamv'], bsf_atl['gphiv'], bsf_atl.isel(time_counter = 0), levels=8, colors='k', linewidths=1, transform=ccrs.PlateCarree(), zorder=2)
ax.set_extent([-85.0, 0.0, 0.0, 80.0], crs=ccrs.PlateCarree())
cl = ax.coastlines()
gl = ax.gridlines(draw_labels=True, linewidth=0.5, linestyle='--')
gl.xlabel_style = {'fontsize': 16}
gl.ylabel_style = {'fontsize': 16}
title = ax.set_title("Annual BSF – 1990", fontsize=18)
plt.colorbar(im, ax=ax, label='Baratropic Stream Function (Sv)')
plt.tight_layout()

def update(frame):
    global cont
    im.set_array((bsf_atl.isel(time_counter = frame)).values.ravel())
    for artist in cont.get_paths():
        pass  
    cont.remove()  
    cont = ax.contour(bsf_atl['glamv'], bsf_atl['gphiv'], bsf_atl.isel(time_counter=frame), levels=8, colors='k', linewidths=1, transform=ccrs.PlateCarree(), zorder=2)
    title.set_text(f"Annual BSF – {1990 + frame}")
    print(frame)
    return [im, title]

plt.close()
anim = FuncAnimation(fig, update, frames=34, interval=500, repeat=True)
anim.save('BSF.gif', writer='pillow', fps=2)
HTML(anim.to_jshtml())



In [ ]:
spg_threshold = -5

# -----------------------------
# 2. Estimate grid-cell area
# -----------------------------
# Assumes bsf_atl has 2D lon/lat coordinates:
#   bsf_atl['glamv'] = longitude
#   bsf_atl['gphiv'] = latitude
#
# If you already have a model grid-cell area variable (e.g. e1v*e2v or e1t*e2t),
# use that instead of this approximation.

R = 6_371_000  # Earth radius in meters

lon = bsf_atl['glamv'].values
lat = bsf_atl['gphiv'].values

# Convert to radians
lon_rad = np.deg2rad(lon)
lat_rad = np.deg2rad(lat)

# Approximate grid spacing from neighboring points
# Works best if lon/lat are 2D curvilinear but reasonably smooth
dlat = np.gradient(lat_rad, axis=0)
dlon = np.gradient(lon_rad, axis=1)

# Cell area approximation: A = R^2 cos(lat) dlon dlat
cell_area = (R**2) * np.cos(lat_rad) * np.abs(dlon) * np.abs(dlat)   # m^2

# Wrap as DataArray with same horizontal dims as bsf_atl
cell_area_da = xr.DataArray(
    cell_area,
    dims=bsf_atl.isel(time_counter=0).dims,
    coords=bsf_atl.isel(time_counter=0).coords
)

# -----------------------------
# 3. Create SPG mask and compute area
# -----------------------------
# SPG mask for each year
spg_mask = bsf_atl <= spg_threshold

# Area of SPG each year
spg_area = (spg_mask * cell_area_da).sum(dim=[d for d in bsf_atl.dims if d != 'time_counter'])

# Convert to million km^2
spg_area_mkm2 = spg_area / 1e12

# -----------------------------
# 4. Build time axis
# -----------------------------
# Option A: if your time coordinate is real datetime
if np.issubdtype(bsf_atl['time_counter'].dtype, np.datetime64):
    years = bsf_atl['time_counter'].dt.year
else:
    # Option B: if frames run from 1990 onward, as in your animation
    years = np.arange(1990, 1990 + bsf_atl.sizes['time_counter'])

# -----------------------------
# 5. Plot
# -----------------------------
plt.figure(figsize=(9, 5))
plt.plot(years, spg_area_mkm2, marker='o')
plt.xlabel('Year', fontsize=12)
plt.ylabel('SPG area (million km$^2$)', fontsize=12)
plt.title(f'Subpolar Gyre area over time\n(BSF ≤ {spg_threshold} Sv)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()